# Multi-Agent Financial Analysis System

AAI-520 Final Team Project, University of San Diego

Group 7: Eric Hernandez, Nolan Robbins, Pranav Dhinakar

GitHub: https://github.com/pdhinaka/AAI-520-NLP-Final-Team-Project-Group7

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from finagent import config
from finagent.agent import research_agent
from finagent.workflows import evaluator_optimizer as eo
from finagent.workflows import prompt_chain, routing

TICKER = "AAPL"

# Chart style: categorical slots from a colorblind-checked palette, muted
# ink for axes and reference lines, light grid.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
INK, MUTED = "#0b0b0b", "#898781"


def style(ax, grid_axis="y"):
    """Recessive axes: no top/right spines, light grid behind data."""
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(MUTED)
    ax.tick_params(colors=MUTED, labelcolor=INK)
    ax.grid(axis=grid_axis, color="#e6e5e0", lw=0.8)
    ax.set_axisbelow(True)
    ax.title.set_ha("left")
    ax.title.set_position((0, 1.02))

## 2. Agent Design and Workflows

_Architecture diagram and design choices._

## 3. Data Tools

Every data source is wrapped as a plain Python function that returns JSON-friendly dicts and is cached on disk. The registry exposes them to the LLM as tools, with each function's docstring used as the tool description.

In [ ]:
from finagent.tools import market_data, registry

print(registry.describe())

In [ ]:
# One-year closing price with the 52-week high and low, the context
# the market specialist reasons about in section 5.
prices = pd.DataFrame(market_data.get_price_history(TICKER, period="1y"))
prices["date"] = pd.to_datetime(prices["date"])
summary = market_data.get_price_summary(TICKER)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(prices["date"], prices["close"], color=SERIES[0], lw=2)
# High label sits above its line and low label below, so neither can
# overlap the price line.
for key, label, va in (("high_52w", "52w high", "bottom"),
                       ("low_52w", "52w low", "top")):
    ax.axhline(summary[key], color=MUTED, ls="--", lw=1)
    ax.text(prices["date"].iloc[0], summary[key],
            f"{label} {summary[key]:.2f}", va=va, color=MUTED,
            fontsize=9)
ax.annotate(f"{summary['last_close']:.2f}",
            (prices["date"].iloc[-1], summary["last_close"]),
            textcoords="offset points", xytext=(4, 0), va="center",
            color=INK, fontsize=9)
ax.margins(y=0.12)
ax.set(ylabel="Close (USD)",
       title=f"{TICKER}: 1-year close, 1y return "
             f"{summary['return_1y']:+.1%}")
style(ax)
plt.show()

## 4. Workflow 1: Prompt Chaining

_Design notes to write: why ingest and preprocess are plain Python, why counts are computed in code, the citation check, and the LLM cache._

### 4.1 Run the chain

In [ ]:
chain = prompt_chain.run_chain(TICKER)
pre = chain["steps"]["preprocess"]
print(
    f"{chain['model']}: ingested {len(chain['steps']['ingest'])} "
    f"articles, kept {len(pre['articles'])} after preprocessing"
)

### 4.2 Preprocess: what was dropped and why

In [ ]:
pd.DataFrame(pre["dropped"])

### 4.3 Classify: topic, sentiment, relevance

In [ ]:
pd.DataFrame([
    {"id": a["id"], "title": a["title"][:60], **a["labels"]}
    for a in chain["steps"]["classify"]
])

### 4.4 Extract: key facts per article

In [ ]:
pd.DataFrame(chain["steps"]["extract"])[
    ["id", "event", "figures", "companies", "dates"]
]

### 4.5 Summarize: the digest

In [ ]:
d = chain["digest"]
display(Markdown(
    f"**{d['headline']}**\n\n{d['summary']}\n\n"
    f"**Catalysts:** {'; '.join(map(str, d['catalysts']))}\n\n"
    f"**Risks:** {'; '.join(map(str, d['risks']))}"
))
print("Sentiment:", d["sentiment_counts"])
print("Topics:", d["topic_counts"])
print("Unknown citations:", d["unknown_citations"] or "none")

### 4.6 Cost and timing per step

In [ ]:
print("Seconds per step:", chain["timings"])
print("Token usage:", chain["usage"])

## 5. Workflow 2: Routing

Routing sends each piece of content about the ticker to one of three specialists (`src/finagent/workflows/routing.py`):

| Specialist | Handles |
| --- | --- |
| earnings | EPS vs. estimates, financial statements, earnings news |
| news | company events: products, legal and regulatory, management |
| market | price action, valuation and analyst targets, macro |

Items are the prompt chain's relevant articles plus one item per structured tool (price summary, company info, financials, earnings, macro). The router works in two stages:

1. **Rules** route structured data by its kind and route articles by the topic label the prompt chain's classify step already assigned. This is free and deterministic, and the reason is easy to state.
2. **LLM router**: only articles labeled `other` go to the model, in one batched call. It returns a specialist and a one-sentence reason for each. An answer naming an unknown specialist falls back to the news analyzer and is marked `default`.

Each specialist gets only its own items. The numbers it reasons over (beat rate, average surprise, revenue growth, margins, position in the 52-week range, upside to the analyst target) are computed in Python. The LLM interprets them and returns a signal, a confidence, and key points. A tool that fails (missing key, rate limit) is skipped and listed in `errors`, so routing still runs on whatever is available.

### 5.1 Route the content

In [ ]:
routed = routing.run_routing(TICKER, chain=chain)
decisions = pd.DataFrame(routed["decisions"])
print("Skipped tools:", routed["errors"] or "none")
decisions[["item_id", "kind", "specialist", "method", "reason"]]

### 5.2 Routing distribution

In [ ]:
# Items per specialist, split by how they were routed. Method gets the
# categorical colors; specialists are on the axis, so color is not the
# only cue.
methods = ["rule", "llm", "default"]
counts = (decisions.groupby(["specialist", "method"]).size()
          .unstack(fill_value=0)
          .reindex(index=list(routing.SPECIALISTS), columns=methods,
                   fill_value=0))

fig, ax = plt.subplots(figsize=(8, 2.8))
left = np.zeros(len(counts))
for color, m in zip(SERIES, methods):
    vals = counts[m].to_numpy()
    ax.barh(counts.index, vals, left=left, color=color, height=0.55,
            edgecolor="white", linewidth=2, label=m)
    for y, (l, v) in enumerate(zip(left, vals)):
        if v:
            ax.text(l + v / 2, y, str(v), ha="center", va="center",
                    color="white", fontsize=9, fontweight="bold")
    left += vals
for y, total in enumerate(left):
    ax.text(total + 0.1, y, f"{int(total)} items", va="center",
            color=INK, fontsize=9)
ax.invert_yaxis()
ax.set(xlabel="Items routed", title=f"{TICKER}: routing by specialist")
ax.set_xlim(0, left.max() * 1.15)
ax.legend(title="method", frameon=False, ncols=3, loc="lower right",
          bbox_to_anchor=(1, 1))
style(ax, grid_axis="x")
plt.show()

### 5.3 Specialist views

In [ ]:
pd.DataFrame([
    {"specialist": name, "items": a["n_items"], "signal": a["signal"],
     "confidence": a["confidence"], "summary": a["summary"]}
    for name, a in routed["analyses"].items()
]).style.set_properties(subset=["summary"], **{"text-align": "left"})

In [ ]:
for name, a in routed["analyses"].items():
    points = "\n".join(f"- {p}" for p in a["key_points"])
    display(Markdown(f"**{name}** ({a['signal']}):\n\n{points}"))
print("Seconds per stage:", routed["timings"])
print("Token usage:", routed["usage"])

## 6. Workflow 3: Evaluator-Optimizer

The evaluator-optimizer turns the three specialist views from routing into one investment brief and improves it in a loop (`src/finagent/workflows/evaluator_optimizer.py`):

1. **Generate**: the generator drafts a brief (view, thesis, evidence, risks, what would change the view) from the specialists' facts and signals.
2. **Evaluate**: a separate, strict "research director" prompt scores the draft from 1 to 5 on five criteria: coverage, evidence, balance, actionability, and clarity. It also writes specific fixes.
3. **Ground**: a sixth criterion, *grounding*, is computed in Python rather than by the LLM. Every number in the draft is matched against the source facts at the precision it was written with, after rescaling (fraction to %, raw to K/M/B/T). Numbers that can't be matched are put at the top of the feedback. The "What would change our view" section is skipped, since its numbers are hypothetical triggers rather than claims. LLM graders are poor at catching invented figures, and a regex is not.
4. **Refine**: the optimizer revises the draft using the feedback. The loop stops when a draft passes or after `max_iters` drafts. To pass, the overall score (mean of the six criteria / 5) must reach the threshold **and** every criterion must score at least `min_score` (default 4), so a strong area can't hide a weak one.

A revision can score lower than the draft it revised, so the loop returns the best-scoring draft, not the last one. All three roles go through the disk cache, so an offline rerun replays identical drafts and scores.

### 6.1 Run the loop

In [ ]:
context = eo.context_from_routing(routed)
loop = eo.run_loop(context, max_iters=3, threshold=0.85)
print(
    f"{len(loop['history'])} drafts; best = iteration "
    f"{loop['best_iteration']}, passed: {loop['passed']}"
)

### 6.2 Scores per iteration

In [ ]:
scores = pd.DataFrame(
    [{"iteration": h["iteration"], **h["scores"],
      "overall": h["overall"], "words": h["words"],
      "ungrounded": len(h["grounding"]["ungrounded"])}
     for h in loop["history"]]
).set_index("iteration")
scores

In [ ]:
# Left: overall score against the pass threshold. Right: each
# criterion per iteration on one sequential scale (darker = higher), so
# the reader sees which criteria the feedback actually moved.
hist = loop["history"]
iters = [h["iteration"] for h in hist]
criteria = list(hist[0]["scores"])
grid = np.array([[h["scores"][c] for h in hist] for c in criteria])

fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1, 1.3]}
)
ax1.plot(iters, [h["overall"] for h in hist], color=SERIES[0], lw=2,
         marker="o", ms=8)
ax1.axhline(loop["threshold"], color=MUTED, ls="--", lw=1)
ax1.text(iters[0], loop["threshold"] + 0.02, "threshold",
         va="bottom", color=MUTED, fontsize=9)
for i, h in zip(iters, hist):
    ax1.annotate(f"{h['overall']:.2f}", (i, h["overall"]),
                 textcoords="offset points", xytext=(0, 9),
                 ha="center", fontsize=9, color=INK)
ax1.set(xticks=iters, xlim=(iters[0] - 0.3, iters[-1] + 0.3),
        ylim=(0, 1.05), xlabel="Iteration",
        title="Overall score")
style(ax1)

ax2.imshow(grid, cmap="Blues", vmin=1, vmax=5, aspect="auto")
ax2.set(xticks=range(len(iters)), xticklabels=iters,
        yticks=range(len(criteria)), yticklabels=criteria,
        xlabel="Iteration", title="Criterion scores (1-5)")
for (r, c), v in np.ndenumerate(grid):
    ax2.text(c, r, f"{v:g}", ha="center", va="center", fontsize=9,
             color="white" if v >= 3.5 else INK)
ax2.tick_params(length=0)
for s in ax2.spines.values():
    s.set_visible(False)
fig.suptitle(f"{TICKER}: evaluator-optimizer", x=0.01, ha="left")
fig.tight_layout()
plt.show()

### 6.3 What the evaluator asked for, and the final brief

In [ ]:
first = loop["history"][0]
print("Iteration 1 feedback:")
for f in first["feedback"]:
    print(" -", f)
print("Ungrounded numbers in draft 1:",
      first["grounding"]["ungrounded"] or "none")
display(Markdown(loop["final"]))

In [ ]:
print("Seconds:", loop["timings"], " Token usage:", loop["usage"])

## 7. Agent Functions

### 7.1 Planning

### 7.2 Dynamic Tool Use

### 7.3 Self-Reflection

### 7.4 Learning Across Runs

## 8. End-to-End Run

## 9. Evaluation and Iteration

## 10. AI Tool Disclosure